# Kiểm tra Recall@10 và Recall@100

Kiểm tra điều kiện Recall@100 > Recall@10 trên SciFact và NFCorpus.

Sử dụng kết quả BM25 Top-100 đã lưu ở phần B2 và bộ chỉ số tự cài trong chi_so.py.

In [1]:
import sys
import json
from pathlib import Path

from beir.datasets.data_loader import GenericDataLoader

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import chi_so

DATA_DIR = PROJECT_ROOT / "phanB" / "du_lieu"
RESULT_DIR = PROJECT_ROOT / "phanB" / "ket_qua"


for name, dataset, filename in [
    ("SciFact", "scifact", "scifact_bm25_top100.json"),
    ("NFCorpus", "nfcorpus", "nfcorpus_bm25_top100.json")
]:
    _, _, qrels = GenericDataLoader(
        data_folder=str(DATA_DIR / dataset)
    ).load(split="test")

    with open(RESULT_DIR / filename, "r", encoding="utf-8") as file:
        results = json.load(file)

    recall_10 = sum(
        chi_so.recall_at_k(
            list(results[qid].keys()), relevance, 10
        )
        for qid, relevance in qrels.items()
    ) / len(qrels)

    recall_100 = sum(
        chi_so.recall_at_k(
            list(results[qid].keys()), relevance, 100
        )
        for qid, relevance in qrels.items()
    ) / len(qrels)

    print(f"{name}:")
    print(f"  Recall@10:  {recall_10:.6f}")
    print(f"  Recall@100: {recall_100:.6f}")
    print(f"  PASS: {recall_100 > recall_10}")

d:\RAG\tep_kem\.venv\Lib\site-packages\beir\datasets\data_loader.py:8: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm
100%|██████████| 5183/5183 [00:00<00:00, 202960.30it/s]


SciFact:
  Recall@10:  0.817278
  Recall@100: 0.928222
  PASS: True


100%|██████████| 3633/3633 [00:00<00:00, 181916.91it/s]

NFCorpus:
  Recall@10:  0.153071
  Recall@100: 0.246407
  PASS: True
